# Day 6 — Capstone: Autonomous Research Assistant

---

Time to build. Today we combine everything from Section 7 into a real, usable **Research Assistant**:

- Give it a question → it searches the web, fetches pages, summarizes, and saves findings
- Uses **LangGraph** for a plan → search → synthesize → save workflow
- Tracks all research in long-term memory (Chroma, from Day 4)
- Bounded by budgets (Day 5)
- Exposed as a FastAPI endpoint with **BackgroundTasks** for async execution


## Architecture

```
   POST /research { "question": "..." }
              │
              ▼
    [FastAPI BackgroundTask]
              │
              ▼
    ┌────────────────────┐
    │ plan_node          │  LLM decides: 1-3 search queries
    └─────────┬──────────┘
              ▼
    ┌────────────────────┐
    │ search_node        │  web_search each query (Tavily)
    └─────────┬──────────┘
              ▼
    ┌────────────────────┐
    │ synthesize_node    │  LLM writes a report from findings
    └─────────┬──────────┘
              ▼
    ┌────────────────────┐
    │ save_node          │  store report in long-term memory
    └─────────┬──────────┘
              ▼
      JOBS[job_id] = report
              │
      GET /research/{job_id}
```

All wrapped in a budget so nothing runs away.


## API surface


### `POST /research`
```json
{"question": "What are the top embedding models in 2026?"}
```
Returns:
```json
{"job_id": "a3f1", "status": "running"}
```

### `GET /research/{job_id}`
```json
{"status": "done", "report": "...", "sources": ["url1", "url2"], "steps": ["planned", ...]}
```

### `GET /research/memory?q=...`
Semantic search over past research reports. Reuses Section 5 patterns.


## Design walkthrough — open `main.py`

The file is organized top-to-bottom:

1. **Setup** — Together, embeddings, Chroma persistent client, Tavily
2. **Tools** — `web_search`, `fetch_url`, `save_report`
3. **Budget** — max_steps, max_tokens (from Day 5)
4. **State** (`ResearchState`) — question, queries, findings, report, sources, budget
5. **Nodes** — `plan_node`, `search_node`, `synthesize_node`, `save_node`
6. **Graph** — linear plan → search → synthesize → save
7. **FastAPI** — `/research`, `/research/{id}`, `/research/memory`

The **cost of an idea** you should take from this file: every long-running LLM app is a graph of a few simple nodes + a state dict + a budget. Nothing more.


## Try it end-to-end

```bash
# 1. Start server
uvicorn main:app --reload

# 2. Fire off a research job
curl -X POST http://localhost:8000/research \
     -H "Content-Type: application/json" \
     -d '{"question":"What are the top open-source embedding models in 2026?"}'
# → {"job_id":"a3f1","status":"running"}

# 3. Poll
curl http://localhost:8000/research/a3f1

# 4. Search past research
curl "http://localhost:8000/research/memory?q=embedding"
```

Or use Swagger at http://localhost:8000/docs.

The whole run typically takes **20–40 seconds** and stays well under $0.02 with Llama-3.3-70B on Together.


## Design decisions worth noting

- **Linear graph, not a ReAct-style tool loop.** For research, the steps are predictable (plan → search → write). A graph reads better than a tool-loop transcript.
- **Budgets are hard walls.** If the plan requests 20 searches, the budget kills the run at step 8. Not "please stop" — a raised exception the executor catches.
- **Findings truncated per source.** We store the first ~500 chars per page. Modern LLMs can eat more, but truncation keeps cost predictable.
- **Long-term memory is per-user.** Even in this single-user demo, we tag every report with `user_id="default"` so we can add multi-tenant later without a schema change.
- **`BackgroundTasks` + in-memory `JOBS` dict.** Simple. If the server restarts, jobs are lost — fine for a portfolio project, not for production. Upgrade path: replace the dict with SQLite or Redis, then switch to Celery for retries.


## What to extend (portfolio polish)

Pick at least one before you call this done:

1. **HITL approval** — before `save_node` writes to memory, require a POST from the user confirming the report.
2. **Streaming intermediate updates** — send server-sent events for each node completion so the UI can show "planning...", "searching...", "writing...".
3. **HTML frontend** — one page that fires the job, polls, and renders the report as Markdown.
4. **Follow-up questions** — after showing a report, let the user ask *"tell me more about X"* and use the past findings as context for the next research run.

Any of these turns the capstone into a **shareable portfolio project**.


## What you built across Section 7

Six days, one autonomous system:

- Understood the **ReAct loop** by writing one from scratch (Day 1)
- Used **function calling** with real tools — search, fetch, calc (Day 2)
- Built stateful workflows with **LangGraph** (Day 3)
- Gave agents **long-term memory** via Chroma (Day 4)
- Added **budgets, retries, HITL** for real-world reliability (Day 5)
- Shipped a **FastAPI-served research assistant** with all of the above (Day 6)

You now have the vocabulary and tools to look at any 2026 "agent" job description and know exactly what they're asking for. Great work — see you in Section 8.
